# **SOTA Beijing Air Pollution Forecasting (GPU Ada CUDA Optimized Version)**
### **CUDA Mixed Precision PyTorch Temporal Attention Net, GPU-Hist XGBoost, CatBoost GPU & High-Speed LightGBM**

**Competition Metric**: Root Mean Squared Error (RMSE)  
**Target Goal**: `< 11.00` (Aiming for `10.0 - 10.8` on Public/Private Leaderboard)  
**Execution Mode**: **NVIDIA Ada / CUDA GPU Ultra-Accelerated Mode**  
**Performance Optimization**: LightGBM OpenMP Thread Capping (`n_jobs=8`) + Full CUDA GPU Acceleration  
**Output File**: `submission_gpu_ada.csv`  

---

In [1]:
# Step 1: Install Dependencies & Setup CUDA GPU Acceleration
#!pip install -q lightgbm xgboost catboost scikit-learn pandas numpy torch matplotlib seaborn scipy joblib

import os
import sys
import math
import gc
import glob
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('ggplot')
warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)

GPU_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("==================================================")
print(" STEP 1: GPU ADA / CUDA ULTRA-FAST DIAGNOSTICS")
print("==================================================")
print(f"[SYSTEM INFO] Python Version:     {sys.version.split()[0]}")
print(f"[SYSTEM INFO] PyTorch Version:    {torch.__version__}")
print(f"[HARDWARE]    CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[GPU HARDWARE] Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"[GPU HARDWARE] Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"[HARDWARE]    Target Device:      {GPU_DEVICE}")
print(f"[OPTIMIZATION] LightGBM CPU Thread Limit: n_jobs=8 (Prevents OpenMP lock contention)")
print("GPU Ada CUDA SOTA libraries successfully initialized!")

 STEP 1: GPU ADA / CUDA ULTRA-FAST DIAGNOSTICS
[SYSTEM INFO] Python Version:     3.12.3
[SYSTEM INFO] PyTorch Version:    2.13.0+cu130
[HARDWARE]    CUDA Available:     True
[GPU HARDWARE] Device Name:      NVIDIA RTX 6000 Ada Generation
[GPU HARDWARE] Total VRAM:       47.37 GB
[HARDWARE]    Target Device:      cuda
[OPTIMIZATION] LightGBM CPU Thread Limit: n_jobs=8 (Prevents OpenMP lock contention)
GPU Ada CUDA SOTA libraries successfully initialized!


In [2]:
# Step 2: Robust Dataset File Resolution
print("==================================================")
print(" STEP 2: DATASET FILE RESOLUTION & INTEGRITY CHECK")
print("==================================================")

def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches:
        return matches[0]
    if os.path.exists(filename):
        return filename
    matches_local = glob.glob(f"./**/{filename}", recursive=True)
    if matches_local:
        return matches_local[0]
    return None

train_path = find_file("train_raw.csv")
test_path = find_file("test.csv")
sample_sub_path = find_file("sample_submission.csv")

print(f"[PATH RESOLUTION] train_raw.csv:       {train_path}")
print(f"[PATH RESOLUTION] test.csv:            {test_path}")
print(f"[PATH RESOLUTION] sample_submission:   {sample_sub_path}")

assert train_path is not None, "Error: train_raw.csv not found!"
assert test_path is not None, "Error: test.csv not found!"

train_size_mb = os.path.getsize(train_path) / (1024 * 1024)
test_size_mb = os.path.getsize(test_path) / (1024 * 1024)
print(f"[VERIFIED] train_raw.csv ({train_size_mb:.2f} MB)")
print(f"[VERIFIED] test.csv ({test_size_mb:.2f} MB)")

 STEP 2: DATASET FILE RESOLUTION & INTEGRITY CHECK
[PATH RESOLUTION] train_raw.csv:       train_raw.csv
[PATH RESOLUTION] test.csv:            test.csv
[PATH RESOLUTION] sample_submission:   None
[VERIFIED] train_raw.csv (26.31 MB)
[VERIFIED] test.csv (6.78 MB)


In [3]:
# Step 3: Domain Physics, Atmospheric Chemistry & Spatial Advection Engine
print("==================================================")
print(" STEP 3: ADVANCED SPATIAL-TEMPORAL FEATURE ENGINE")
print("==================================================")

WD_MAP = {
    'N': 0.0, 'NNE': 22.5, 'NE': 45.0, 'ENE': 67.5,
    'E': 90.0, 'ESE': 112.5, 'SE': 135.0, 'SSE': 157.5,
    'S': 180.0, 'SSW': 202.5, 'SW': 225.0, 'WSW': 247.5,
    'W': 270.0, 'WNW': 292.5, 'NW': 315.0, 'NNW': 337.5
}

POLLUTANTS = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
METEO = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

def add_physics_and_chemistry_features(df):
    df = df.copy()
    
    if 'date' not in df.columns and set(['year', 'month', 'day', 'hour']).issubset(df.columns):
        df['date'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
        
    if 'date' in df.columns:
        df['dayofweek'] = df['date'].dt.dayofweek
        df['dayofyear'] = df['date'].dt.dayofyear
    else:
        df['dayofweek'] = 0
        df['dayofyear'] = 1
        
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7.0)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7.0)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    df['is_heating_season'] = df['month'].isin([11, 12, 1, 2, 3]).astype(np.float32)

    if 'wd' in df.columns:
        wd_deg = df['wd'].map(WD_MAP).fillna(0.0)
    else:
        wd_deg = 0.0
    wd_rad = np.radians(wd_deg)
    wspm = df['WSPM'].fillna(0.0)
    df['Wx'] = wspm * np.sin(wd_rad)
    df['Wy'] = wspm * np.cos(wd_rad)

    temp_c = df['TEMP'].fillna(0.0)
    dew_c = df['DEWP'].fillna(0.0)
    df['dew_point_depression'] = temp_c - dew_c
    rh = 100.0 * np.exp((17.625 * dew_c)/(243.04 + dew_c) - (17.625 * temp_c)/(243.04 + temp_c))
    df['relative_humidity'] = np.clip(rh, 0.0, 100.0)
    df['ventilation_index'] = wspm * (df['dew_point_depression'] + 10.0)
    df['vapor_pressure_deficit'] = (100.0 - df['relative_humidity']) * np.exp(17.27 * temp_c / (temp_c + 237.7)) / 100.0

    pm10 = df['PM10'].fillna(0.0)
    pm25 = df['PM2.5'].fillna(0.0)
    df['coarse_pm'] = np.maximum(pm10 - pm25, 0.0)
    df['PM25_PM10_ratio'] = pm25 / (pm10 + 1.0)
    df['coarse_ratio'] = df['coarse_pm'] / (pm10 + 1.0)
    df['PM25_CO_ratio'] = pm25 / (df['CO'].fillna(0.0) / 1000.0 + 1.0)
    df['NO2_O3_ratio'] = df['NO2'].fillna(0.0) / (df['O3'].fillna(0.0) + 1.0)
    df['SO2_CO_ratio'] = df['SO2'].fillna(0.0) / (df['CO'].fillna(0.0) / 1000.0 + 1.0)
    df['total_pollution'] = pm25 + pm10 + df['SO2'].fillna(0.0) + df['NO2'].fillna(0.0) + (df['CO'].fillna(0.0) / 1000.0) + df['O3'].fillna(0.0)

    return df

print("[FEATURE LOG] Feature Engineering logic compiled successfully:")
print("  - Atmospheric Science: Magnus RH, Dew Depression, Ventilation Index")
print("  - Chemical & Aerosol Ratios: PM2.5/PM10, Coarse PM, SO2/CO, Photochemical NO2/O3")
print("  - Vector Advection: Wx, Wy Wind Components")
print("  - Trigonometric Cyclical Calendar Encodings")

 STEP 3: ADVANCED SPATIAL-TEMPORAL FEATURE ENGINE
[FEATURE LOG] Feature Engineering logic compiled successfully:
  - Atmospheric Science: Magnus RH, Dew Depression, Ventilation Index
  - Chemical & Aerosol Ratios: PM2.5/PM10, Coarse PM, SO2/CO, Photochemical NO2/O3
  - Vector Advection: Wx, Wy Wind Components
  - Trigonometric Cyclical Calendar Encodings


In [4]:
# Step 4: High-Performance Train & Test Dataset Builder
print("==================================================")
print(" STEP 4: BUILDING TRAIN & TEST FEATURE MATRICES")
print("==================================================")

def build_training_matrix(train_csv_path):
    print(f"[TRAIN BUILDER] Loading raw training CSV: {train_csv_path}...")
    df_raw = pd.read_csv(train_csv_path)
    df_raw['date'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])
    df_raw = df_raw.sort_values(by=['station', 'date']).reset_index(drop=True)

    stations = df_raw['station'].unique()
    print(f"[TRAIN BUILDER] Found {len(stations)} stations across {len(df_raw):,} records.")

    processed_station_dfs = []
    for i, (station, group) in enumerate(df_raw.groupby('station')):
        print(f"  -> [{i+1:02d}/{len(stations):02d}] Processing station '{station:15s}' ({len(group):,} rows)...")
        df_p = add_physics_and_chemistry_features(group)
        interp_cols = [c for c in df_p.columns if df_p[c].dtype in [np.float64, np.float32, np.int64]]
        df_p[interp_cols] = df_p[interp_cols].interpolate(method='linear', limit_direction='both').ffill().bfill()
        processed_station_dfs.append(df_p)

    print("[TRAIN BUILDER] Merging station DataFrames and computing regional spatial advection...")
    df_full = pd.concat(processed_station_dfs, ignore_index=True)

    spatial_aggs = df_full.groupby('date').agg({
        'PM2.5': ['mean', 'std', 'max', 'min'],
        'PM10': ['mean'],
        'NO2': ['mean'],
        'SO2': ['mean'],
        'CO': ['mean'],
        'O3': ['mean'],
        'TEMP': ['mean'],
        'PRES': ['mean'],
        'WSPM': ['mean']
    })
    spatial_aggs.columns = ['_'.join(c).strip() + '_spatial' for c in spatial_aggs.columns]
    df_full = df_full.merge(spatial_aggs, on='date', how='left')

    df_full['PM2.5_diff_spatial_mean'] = df_full['PM2.5'] - df_full['PM2.5_mean_spatial']
    df_full['advection_x'] = df_full['Wx'] * df_full['PM2.5_mean_spatial']
    df_full['advection_y'] = df_full['Wy'] * df_full['PM2.5_mean_spatial']

    print("[TRAIN BUILDER] Constructing station-wise lag features, rolling statistics, and EMAs...")
    final_station_dfs = []
    for station, df_st in df_full.groupby('station'):
        df_st = df_st.copy()
        
        df_st['PM2.5_ema_3'] = df_st['PM2.5'].ewm(span=3, adjust=False).mean()
        df_st['PM2.5_ema_6'] = df_st['PM2.5'].ewm(span=6, adjust=False).mean()
        df_st['PM2.5_ema_12'] = df_st['PM2.5'].ewm(span=12, adjust=False).mean()

        for lag in range(1, 25):
            df_st[f'PM2.5_lag_{lag}'] = df_st['PM2.5'].shift(lag)

        for k in [1, 2, 3, 4, 6, 12, 24]:
            df_st[f'PM2.5_diff_{k}'] = df_st['PM2.5'] - df_st['PM2.5'].shift(k)
            df_st[f'PM2.5_diff_{k}_ratio'] = (df_st['PM2.5'] - df_st['PM2.5'].shift(k)) / (df_st['PM2.5'].shift(k) + 1.0)

        df_st['PM2.5_accel'] = (df_st['PM2.5'] - df_st['PM2.5'].shift(1)) - (df_st['PM2.5'].shift(1) - df_st['PM2.5'].shift(2))

        for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'SO2', 'TEMP', 'PRES', 'DEWP', 'WSPM', 'Wx', 'Wy', 'relative_humidity', 'dew_point_depression']:
            for k in [1, 3, 6]:
                df_st[f'{col}_diff_{k}'] = df_st[col] - df_st[col].shift(k)
                df_st[f'{col}_lag_{k}'] = df_st[col].shift(k)

        for w in [3, 6, 12, 24]:
            df_st[f'PM2.5_roll_mean_{w}h'] = df_st['PM2.5'].rolling(w, min_periods=1).mean()
            df_st[f'PM2.5_roll_std_{w}h'] = df_st['PM2.5'].rolling(w, min_periods=1).std().fillna(0)
            df_st[f'PM2.5_roll_min_{w}h'] = df_st['PM2.5'].rolling(w, min_periods=1).min()
            df_st[f'PM2.5_roll_max_{w}h'] = df_st['PM2.5'].rolling(w, min_periods=1).max()

        df_st['PM2.5_roll_range_24h'] = df_st['PM2.5_roll_max_24h'] - df_st['PM2.5_roll_min_24h']
        df_st['PM2.5_dev_mean_6h'] = df_st['PM2.5'] - df_st['PM2.5_roll_mean_6h']
        df_st['PM2.5_dev_mean_24h'] = df_st['PM2.5'] - df_st['PM2.5_roll_mean_24h']
        df_st['PM2.5_ratio_mean_24h'] = df_st['PM2.5'] / (df_st['PM2.5_roll_mean_24h'] + 1.0)

        for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'TEMP', 'WSPM', 'relative_humidity']:
            for w in [3, 6, 12]:
                df_st[f'{col}_roll_mean_{w}h'] = df_st[col].rolling(w, min_periods=1).mean()
                df_st[f'{col}_roll_std_{w}h'] = df_st[col].rolling(w, min_periods=1).std().fillna(0)

        df_st['target'] = df_st['PM2.5'].shift(-1)
        final_station_dfs.append(df_st)

    df_train_full = pd.concat(final_station_dfs, ignore_index=True)
    df_train_clean = df_train_full.dropna(subset=['target', 'PM2.5_lag_24']).reset_index(drop=True)
    print(f"[SUCCESS] Training Dataset created: {len(df_train_clean):,} sliding window samples.")
    return df_train_clean

def build_test_matrix(test_csv_path):
    print(f"\n[TEST BUILDER] Loading raw test CSV: {test_csv_path}...")
    df_test_raw = pd.read_csv(test_csv_path)
    total_test_rows = len(df_test_raw)
    print(f"[TEST BUILDER] Unflattening {total_test_rows:,} test windows into continuous sequence matrices...")

    feature_vars = ['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM']
    test_processed_rows = []
    
    for idx, row in df_test_raw.iterrows():
        if (idx + 1) % 1000 == 0 or (idx + 1) == total_test_rows:
            print(f"  -> Processing Test Row [{idx+1:04d} / {total_test_rows:04d}] ({(idx+1)/total_test_rows*100:5.1f}%)...")

        station = row['station']
        row_id = row['id']

        mini_records = []
        for lag in range(24, 0, -1):
            rec = {var: row[f'{var}_lag_{lag}'] for var in feature_vars}
            mini_records.append(rec)

        df_mini = pd.DataFrame(mini_records)
        df_mini['station'] = station
        df_mini_p = add_physics_and_chemistry_features(df_mini)
        
        num_cols = [c for c in df_mini_p.columns if df_mini_p[c].dtype in [np.float64, np.float32, np.int64]]
        df_mini_p[num_cols] = df_mini_p[num_cols].interpolate(method='linear', limit_direction='both').ffill().bfill()

        df_mini_p['PM2.5_mean_spatial'] = df_mini_p['PM2.5']
        df_mini_p['PM2.5_std_spatial'] = 0.0
        df_mini_p['PM2.5_max_spatial'] = df_mini_p['PM2.5']
        df_mini_p['PM2.5_min_spatial'] = df_mini_p['PM2.5']
        df_mini_p['PM10_mean_spatial'] = df_mini_p['PM10']
        df_mini_p['NO2_mean_spatial'] = df_mini_p['NO2']
        df_mini_p['SO2_mean_spatial'] = df_mini_p['SO2']
        df_mini_p['CO_mean_spatial'] = df_mini_p['CO']
        df_mini_p['O3_mean_spatial'] = df_mini_p['O3']
        df_mini_p['TEMP_mean_spatial'] = df_mini_p['TEMP']
        df_mini_p['PRES_mean_spatial'] = df_mini_p['PRES']
        df_mini_p['WSPM_mean_spatial'] = df_mini_p['WSPM']

        df_mini_p['PM2.5_diff_spatial_mean'] = 0.0
        df_mini_p['advection_x'] = df_mini_p['Wx'] * df_mini_p['PM2.5']
        df_mini_p['advection_y'] = df_mini_p['Wy'] * df_mini_p['PM2.5']

        df_mini_p['PM2.5_ema_3'] = df_mini_p['PM2.5'].ewm(span=3, adjust=False).mean()
        df_mini_p['PM2.5_ema_6'] = df_mini_p['PM2.5'].ewm(span=6, adjust=False).mean()
        df_mini_p['PM2.5_ema_12'] = df_mini_p['PM2.5'].ewm(span=12, adjust=False).mean()

        for lag in range(1, 25):
            df_mini_p[f'PM2.5_lag_{lag}'] = df_mini_p['PM2.5'].shift(lag)

        for k in [1, 2, 3, 4, 6, 12, 24]:
            df_mini_p[f'PM2.5_diff_{k}'] = df_mini_p['PM2.5'] - df_mini_p['PM2.5'].shift(k)
            df_mini_p[f'PM2.5_diff_{k}_ratio'] = (df_mini_p['PM2.5'] - df_mini_p['PM2.5'].shift(k)) / (df_mini_p['PM2.5'].shift(k) + 1.0)

        df_mini_p['PM2.5_accel'] = (df_mini_p['PM2.5'] - df_mini_p['PM2.5'].shift(1)) - (df_mini_p['PM2.5'].shift(1) - df_mini_p['PM2.5'].shift(2))

        for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'SO2', 'TEMP', 'PRES', 'DEWP', 'WSPM', 'Wx', 'Wy', 'relative_humidity', 'dew_point_depression']:
            for k in [1, 3, 6]:
                df_mini_p[f'{col}_diff_{k}'] = df_mini_p[col] - df_mini_p[col].shift(k)
                df_mini_p[f'{col}_lag_{k}'] = df_mini_p[col].shift(k)

        for w in [3, 6, 12, 24]:
            df_mini_p[f'PM2.5_roll_mean_{w}h'] = df_mini_p['PM2.5'].rolling(w, min_periods=1).mean()
            df_mini_p[f'PM2.5_roll_std_{w}h'] = df_mini_p['PM2.5'].rolling(w, min_periods=1).std().fillna(0)
            df_mini_p[f'PM2.5_roll_min_{w}h'] = df_mini_p['PM2.5'].rolling(w, min_periods=1).min()
            df_mini_p[f'PM2.5_roll_max_{w}h'] = df_mini_p['PM2.5'].rolling(w, min_periods=1).max()

        df_mini_p['PM2.5_roll_range_24h'] = df_mini_p['PM2.5_roll_max_24h'] - df_mini_p['PM2.5_roll_min_24h']
        df_mini_p['PM2.5_dev_mean_6h'] = df_mini_p['PM2.5'] - df_mini_p['PM2.5_roll_mean_6h']
        df_mini_p['PM2.5_dev_mean_24h'] = df_mini_p['PM2.5'] - df_mini_p['PM2.5_roll_mean_24h']
        df_mini_p['PM2.5_ratio_mean_24h'] = df_mini_p['PM2.5'] / (df_mini_p['PM2.5_roll_mean_24h'] + 1.0)

        for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'TEMP', 'WSPM', 'relative_humidity']:
            for w in [3, 6, 12]:
                df_mini_p[f'{col}_roll_mean_{w}h'] = df_mini_p[col].rolling(w, min_periods=1).mean()
                df_mini_p[f'{col}_roll_std_{w}h'] = df_mini_p[col].rolling(w, min_periods=1).std().fillna(0)

        last_row = df_mini_p.iloc[-1].to_dict()
        last_row['id'] = row_id
        last_row['station'] = station
        test_processed_rows.append(last_row)

    df_test_proc = pd.DataFrame(test_processed_rows)
    print(f"[SUCCESS] Test Dataset created: {len(df_test_proc):,} test target rows.")
    return df_test_proc

df_train = build_training_matrix(train_path)
df_test = build_test_matrix(test_path)

 STEP 4: BUILDING TRAIN & TEST FEATURE MATRICES
[TRAIN BUILDER] Loading raw training CSV: train_raw.csv...
[TRAIN BUILDER] Found 12 stations across 315,648 records.
  -> [01/12] Processing station 'Aotizhongxin   ' (26,304 rows)...
  -> [02/12] Processing station 'Changping      ' (26,304 rows)...
  -> [03/12] Processing station 'Dingling       ' (26,304 rows)...
  -> [04/12] Processing station 'Dongsi         ' (26,304 rows)...
  -> [05/12] Processing station 'Guanyuan       ' (26,304 rows)...
  -> [06/12] Processing station 'Gucheng        ' (26,304 rows)...
  -> [07/12] Processing station 'Huairou        ' (26,304 rows)...
  -> [08/12] Processing station 'Nongzhanguan   ' (26,304 rows)...
  -> [09/12] Processing station 'Shunyi         ' (26,304 rows)...
  -> [10/12] Processing station 'Tiantan        ' (26,304 rows)...
  -> [11/12] Processing station 'Wanliu         ' (26,304 rows)...
  -> [12/12] Processing station 'Wanshouxigong  ' (26,304 rows)...
[TRAIN BUILDER] Merging station

In [5]:
# Step 5: Feature Selection, Target Encoding & Group Split Preparation
print("==================================================")
print(" STEP 5: PREPARING FEATURE MATRICES & TEMPORAL GROUPS")
print("==================================================")

le_station = LabelEncoder()
df_train['station_cat'] = le_station.fit_transform(df_train['station'])
df_test['station_cat'] = le_station.transform(df_test['station'])

station_means = df_train.groupby('station_cat')['target'].mean().to_dict()
station_stds = df_train.groupby('station_cat')['target'].std().to_dict()
df_train['station_target_mean'] = df_train['station_cat'].map(station_means)
df_train['station_target_std'] = df_train['station_cat'].map(station_stds)
df_test['station_target_mean'] = df_test['station_cat'].map(station_means)
df_test['station_target_std'] = df_test['station_cat'].map(station_stds)

df_train['year_month_group'] = df_train['year'].astype(str) + '_' + df_train['month'].astype(str)
groups = df_train['year_month_group'].values

drop_cols = ['No', 'year', 'month', 'day', 'hour', 'date', 'wd', 'station', 'target', 'id', 'year_month_group']
feature_cols = [c for c in df_train.columns if c not in drop_cols]

X = df_train[feature_cols].copy().fillna(0).astype(np.float32)
y_raw = df_train['target'].copy().values.astype(np.float32)
y_log1p = np.log1p(np.maximum(y_raw, 0.0))

X_test = df_test[feature_cols].copy().fillna(0).astype(np.float32)

print(f"[MATRIX DIAGNOSTICS] Total Train Samples (X):       {X.shape[0]:,}")
print(f"[MATRIX DIAGNOSTICS] Total SOTA Feature Count (X):   {X.shape[1]:,}")
print(f"[MATRIX DIAGNOSTICS] Total Test Samples (X_test):   {X_test.shape[0]:,}")
print(f"[MATRIX DIAGNOSTICS] Unique Temporal Groups:        {len(np.unique(groups))} ({', '.join(np.unique(groups)[:5])}...)")
print(f"[TARGET STATS]       Target Mean (PM2.5):           {y_raw.mean():.2f} ug/m3")
print(f"[TARGET STATS]       Target Min / Max:              {y_raw.min():.2f} / {y_raw.max():.2f} ug/m3")

 STEP 5: PREPARING FEATURE MATRICES & TEMPORAL GROUPS
[MATRIX DIAGNOSTICS] Total Train Samples (X):       315,348
[MATRIX DIAGNOSTICS] Total SOTA Feature Count (X):   235
[MATRIX DIAGNOSTICS] Total Test Samples (X_test):   4,103
[MATRIX DIAGNOSTICS] Unique Temporal Groups:        36 (2013_10, 2013_11, 2013_12, 2013_3, 2013_4...)
[TARGET STATS]       Target Mean (PM2.5):           80.48 ug/m3
[TARGET STATS]       Target Min / Max:              2.00 / 999.00 ug/m3


In [6]:
# Step 6: PyTorch Deep Temporal Attention 1D-CNN + BiGRU Architecture
print("==================================================")
print(" STEP 6: PYTORCH DEEP TEMPORAL ATTENTION NEURAL NET")
print("==================================================")

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x):
        scores = self.attn(x)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(x * weights, dim=1)
        return context

class DeepTemporalAttention_Net(nn.Module):
    def __init__(self, input_dim):
        super(DeepTemporalAttention_Net, self).__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        
        self.conv1 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1, dilation=1)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=2, dilation=2)
        self.ln_conv = nn.LayerNorm(128)
        self.gelu = nn.GELU()
        
        self.bigru = nn.GRU(128, 64, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.attention = TemporalAttention(128)
        
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        h = self.in_proj(x)
        h_seq = h.unsqueeze(1)
        
        h_conv = h_seq.transpose(1, 2)
        c1 = self.gelu(self.conv1(h_conv))
        c2 = self.conv2(c1)
        h_conv = self.gelu(self.ln_conv((c2 + h_conv).transpose(1, 2)))
        
        out_gru, _ = self.bigru(h_conv)
        context = self.attention(out_gru)
        out = self.head(context)
        return out.squeeze(-1)

sample_net = DeepTemporalAttention_Net(X.shape[1])
total_params = sum(p.numel() for p in sample_net.parameters() if p.requires_grad)
print(f"[MODEL INIT] DeepTemporalAttention_Net initialized | Trainable Parameters: {total_params:,}")

 STEP 6: PYTORCH DEEP TEMPORAL ATTENTION NEURAL NET
[MODEL INIT] DeepTemporalAttention_Net initialized | Trainable Parameters: 297,090


In [7]:
# Step 7: 5-Fold GroupKFold Cross-Validation Training (GPU CUDA High-Speed Optimized)
print("==================================================")
print(" STEP 7: 5-FOLD TEMPORAL GROUP GPU CUDA TRAINING")
print("==================================================")

save_dir = 'saved_models_gpu'
os.makedirs(save_dir, exist_ok=True)

oof_lgb = np.zeros(len(df_train))
oof_xgb = np.zeros(len(df_train))
oof_cb  = np.zeros(len(df_train))
oof_nn  = np.zeros(len(df_train))

preds_lgb = np.zeros(len(df_test))
preds_xgb = np.zeros(len(df_test))
preds_cb  = np.zeros(len(df_test))
preds_nn  = np.zeros(len(df_test))

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_log1p, groups=groups)):
    print(f"\n" + "="*65)
    print(f"   >>> STARTING GROUP FOLD {fold+1} / 5 (GPU ULTRA-FAST MODE) <<<")
    print("="*65)
    X_tr, y_tr_log = X.iloc[train_idx], y_log1p[train_idx]
    X_va, y_va_log = X.iloc[val_idx], y_log1p[val_idx]
    y_va_raw = y_raw[val_idx]
    print(f"[FOLD {fold+1} INFO] Train samples: {len(X_tr):,} | Validation samples: {len(X_va):,}")
    
    # 1. LIGHTGBM (Log Target Scale, Capped n_jobs=8 for Maximum CPU Bandwidth)
    print(f"\n[FOLD {fold+1}] [1/4] Training High-Speed LightGBM (n_jobs=8)...")
    model_lgb = lgb.LGBMRegressor(
        n_estimators=3000, learning_rate=0.03, num_leaves=63, max_depth=8,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30, objective='regression', metric='rmse',
        random_state=SEED + fold, n_jobs=8, device='cpu'
    )
    model_lgb.fit(
        X_tr, y_tr_log,
        eval_set=[(X_va, y_va_log)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(period=200)]
    )
    
    val_pred_log = np.clip(model_lgb.predict(X_va), -10.0, 7.0)
    oof_lgb[val_idx] = np.expm1(val_pred_log)
    test_lgb_log = np.clip(model_lgb.predict(X_test), -10.0, 7.0)
    preds_lgb += np.expm1(test_lgb_log) / 5.0
    lgb_score = np.sqrt(mean_squared_error(y_va_raw, oof_lgb[val_idx]))
    print(f"===> [FOLD {fold+1} RESULT] LightGBM Validation Raw RMSE: {lgb_score:.4f} ug/m3")
    
    # 2. XGBOOST (CUDA GPU Hist)
    print(f"\n[FOLD {fold+1}] [2/4] Training XGBoost (CUDA GPU)...")
    model_xgb = xgb.XGBRegressor(
        n_estimators=3000, learning_rate=0.03, max_depth=7,
        subsample=0.8, colsample_bytree=0.7, gamma=0.1, reg_alpha=1.0, reg_lambda=2.0,
        early_stopping_rounds=100, objective='reg:squarederror', eval_metric='rmse',
        random_state=SEED + fold, tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    model_xgb.fit(X_tr, y_tr_log, eval_set=[(X_va, y_va_log)], verbose=200)
    val_pred_xgb_log = np.clip(model_xgb.predict(X_va), -10.0, 7.0)
    oof_xgb[val_idx] = np.expm1(val_pred_xgb_log)
    test_xgb_log = np.clip(model_xgb.predict(X_test), -10.0, 7.0)
    preds_xgb += np.expm1(test_xgb_log) / 5.0
    xgb_score = np.sqrt(mean_squared_error(y_va_raw, oof_xgb[val_idx]))
    print(f"===> [FOLD {fold+1} RESULT] XGBoost Validation Raw RMSE:  {xgb_score:.4f} ug/m3")
    
    # 3. CATBOOST (GPU Task Type)
    print(f"\n[FOLD {fold+1}] [3/4] Training CatBoost (GPU)...")
    cb_task = 'GPU' if torch.cuda.is_available() else 'CPU'
    model_cb = cb.CatBoostRegressor(
        iterations=3000, learning_rate=0.03, depth=7, loss_function='RMSE', eval_metric='RMSE',
        task_type=cb_task, random_seed=SEED + fold, verbose=200
    )
    model_cb.fit(X_tr, y_tr_log, eval_set=(X_va, y_va_log), early_stopping_rounds=100)
    val_pred_cb_log = np.clip(model_cb.predict(X_va), -10.0, 7.0)
    oof_cb[val_idx] = np.expm1(val_pred_cb_log)
    test_cb_log = np.clip(model_cb.predict(X_test), -10.0, 7.0)
    preds_cb += np.expm1(test_cb_log) / 5.0
    cb_score = np.sqrt(mean_squared_error(y_va_raw, oof_cb[val_idx]))
    print(f"===> [FOLD {fold+1} RESULT] CatBoost Validation Raw RMSE: {cb_score:.4f} ug/m3")
    
    # 4. PYTORCH DEEP TEMPORAL ATTENTION NEURAL NET (GPU CUDA Accelerated)
    print(f"\n[FOLD {fold+1}] [4/4] Training PyTorch Deep Temporal Attention Net (GPU CUDA)...")
    scaler_x = StandardScaler()
    X_tr_sc = scaler_x.fit_transform(X_tr)
    X_va_sc = scaler_x.transform(X_va)
    X_te_sc = scaler_x.transform(X_test)
    
    ds_tr = TensorDataset(torch.tensor(X_tr_sc, dtype=torch.float32), torch.tensor(y_tr_log, dtype=torch.float32))
    ds_va = TensorDataset(torch.tensor(X_va_sc, dtype=torch.float32), torch.tensor(y_va_log, dtype=torch.float32))
    
    dl_tr = DataLoader(ds_tr, batch_size=1024, shuffle=True, pin_memory=True if torch.cuda.is_available() else False)
    dl_va = DataLoader(ds_va, batch_size=2048, shuffle=False)
    
    net = DeepTemporalAttention_Net(X_tr.shape[1]).to(GPU_DEVICE)
    optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)
    criterion = nn.MSELoss()
    
    best_loss = float('inf')
    best_preds_raw = None
    
    for epoch in range(1, 21):
        net.train()
        running_loss = 0.0
        for bx, by in dl_tr:
            bx, by = bx.to(GPU_DEVICE), by.to(GPU_DEVICE)
            optimizer.zero_grad()
            loss = criterion(net(bx), by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(by)
            
        train_epoch_loss = running_loss / len(ds_tr)
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        
        net.eval()
        val_preds_list = []
        with torch.no_grad():
            for bx, by in dl_va:
                val_preds_list.append(net(bx.to(GPU_DEVICE)).cpu().numpy())
        val_preds_log = np.clip(np.concatenate(val_preds_list), -10.0, 7.0)
        val_preds_raw = np.expm1(val_preds_log)
        val_rmse = np.sqrt(mean_squared_error(y_va_raw, val_preds_raw))
        
        print(f"  [PyTorch Epoch {epoch:02d}/20] Train Loss: {train_epoch_loss:.5f} | Val RMSE (Raw Scale): {val_rmse:6.4f} ug/m3 | LR: {current_lr:.6f}")
            
        if val_rmse < best_loss:
            best_loss = val_rmse
            best_preds_raw = val_preds_raw
            torch.save(net.state_dict(), os.path.join(save_dir, f'pytorch_nn_fold_{fold+1}.pt'))
            
    oof_nn[val_idx] = best_preds_raw
    net.eval()
    with torch.no_grad():
        test_tensor = torch.tensor(X_te_sc, dtype=torch.float32).to(GPU_DEVICE)
        te_log = np.clip(net(test_tensor).cpu().numpy(), -10.0, 7.0)
        preds_nn += np.expm1(te_log) / 5.0
    print(f"===> [FOLD {fold+1} RESULT] PyTorch Deep Net Best Validation Raw RMSE: {best_loss:.4f} ug/m3")
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

 STEP 7: 5-FOLD TEMPORAL GROUP GPU CUDA TRAINING

   >>> STARTING GROUP FOLD 1 / 5 (GPU ULTRA-FAST MODE) <<<
[FOLD 1 INFO] Train samples: 253,716 | Validation samples: 61,632

[FOLD 1] [1/4] Training High-Speed LightGBM (n_jobs=8)...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073360 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 55602
[LightGBM] [Info] Number of data points in the train set: 253716, number of used features: 235
[LightGBM] [Info] Start training from score 3.927807
[200]	valid_0's rmse: 0.269005
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

0:	learn: 1.0633381	test: 1.0470910	best: 1.0470910 (0)	total: 489ms	remaining: 24m 27s
200:	learn: 0.2747467	test: 0.2756217	best: 0.2756217 (200)	total: 40.4s	remaining: 9m 22s
400:	learn: 0.2665724	test: 0.2705991	best: 0.2705991 (400)	total: 1m 20s	remaining: 8m 40s
600:	learn: 0.2618733	test: 0.2687490	best: 0.2687490 (600)	total: 2m	remaining: 8m 1s
800:	learn: 0.2583026	test: 0.2676569	best: 0.2676569 (800)	total: 2m 40s	remaining: 7m 19s
1000:	learn: 0.2553000	test: 0.2669981	best: 0.2669981 (1000)	total: 3m 21s	remaining: 6m 42s
1200:	learn: 0.2526574	test: 0.2665388	best: 0.2665388 (1200)	total: 4m 2s	remaining: 6m 2s
1400:	learn: 0.2502729	test: 0.2661464	best: 0.2661426 (1399)	total: 4m 41s	remaining: 5m 21s
1600:	learn: 0.2481109	test: 0.2658552	best: 0.2658552 (1600)	total: 5m 21s	remaining: 4m 40s
1800:	learn: 0.2460576	test: 0.2656403	best: 0.2656403 (1800)	total: 6m 1s	remaining: 4m
2000:	learn: 0.2441155	test: 0.2654416	best: 0.2654388 (1997)	total: 6m 46s	remaining: 

0:	learn: 1.0629816	test: 1.0471758	best: 1.0471758 (0)	total: 292ms	remaining: 14m 36s
200:	learn: 0.2705714	test: 0.2917275	best: 0.2917275 (200)	total: 1m 1s	remaining: 14m 14s
400:	learn: 0.2624256	test: 0.2863545	best: 0.2863545 (400)	total: 2m 4s	remaining: 13m 28s
600:	learn: 0.2579288	test: 0.2844660	best: 0.2844660 (600)	total: 3m 4s	remaining: 12m 15s
800:	learn: 0.2545938	test: 0.2835565	best: 0.2835565 (800)	total: 4m 3s	remaining: 11m 8s
1000:	learn: 0.2517882	test: 0.2828582	best: 0.2828582 (1000)	total: 5m 1s	remaining: 10m 2s
1200:	learn: 0.2492887	test: 0.2822985	best: 0.2822919 (1199)	total: 6m	remaining: 8m 59s
1400:	learn: 0.2470285	test: 0.2819466	best: 0.2819466 (1400)	total: 6m 59s	remaining: 7m 58s
1600:	learn: 0.2448997	test: 0.2815915	best: 0.2815915 (1600)	total: 8m 3s	remaining: 7m 2s
1800:	learn: 0.2428556	test: 0.2813165	best: 0.2813146 (1799)	total: 9m 2s	remaining: 6m 1s
2000:	learn: 0.2410844	test: 0.2810884	best: 0.2810884 (2000)	total: 10m 3s	remainin

0:	learn: 1.0573071	test: 1.0719233	best: 1.0719233 (0)	total: 280ms	remaining: 13m 59s
200:	learn: 0.2739025	test: 0.2809426	best: 0.2809426 (200)	total: 59.6s	remaining: 13m 50s
400:	learn: 0.2655072	test: 0.2753552	best: 0.2753552 (400)	total: 1m 59s	remaining: 12m 52s
600:	learn: 0.2608233	test: 0.2732014	best: 0.2732014 (600)	total: 2m 58s	remaining: 11m 51s
800:	learn: 0.2571152	test: 0.2717620	best: 0.2717620 (800)	total: 3m 57s	remaining: 10m 51s
1000:	learn: 0.2541217	test: 0.2708640	best: 0.2708640 (1000)	total: 4m 56s	remaining: 9m 51s
1200:	learn: 0.2514205	test: 0.2702485	best: 0.2702446 (1195)	total: 5m 55s	remaining: 8m 52s
1400:	learn: 0.2490613	test: 0.2697912	best: 0.2697866 (1399)	total: 6m 54s	remaining: 7m 53s
1600:	learn: 0.2469047	test: 0.2693748	best: 0.2693748 (1600)	total: 7m 53s	remaining: 6m 53s
1800:	learn: 0.2448808	test: 0.2689880	best: 0.2689880 (1799)	total: 8m 51s	remaining: 5m 53s
2000:	learn: 0.2429362	test: 0.2686813	best: 0.2686812 (1999)	total: 9m

0:	learn: 1.0612396	test: 1.0554279	best: 1.0554279 (0)	total: 284ms	remaining: 14m 11s
200:	learn: 0.2749447	test: 0.2743848	best: 0.2743848 (200)	total: 59.7s	remaining: 13m 51s
400:	learn: 0.2665683	test: 0.2693864	best: 0.2693864 (400)	total: 1m 58s	remaining: 12m 48s
600:	learn: 0.2616936	test: 0.2673767	best: 0.2673767 (600)	total: 2m 56s	remaining: 11m 46s
800:	learn: 0.2580348	test: 0.2661417	best: 0.2661417 (800)	total: 3m 56s	remaining: 10m 48s
1000:	learn: 0.2550458	test: 0.2652718	best: 0.2652718 (1000)	total: 4m 56s	remaining: 9m 52s
1200:	learn: 0.2524623	test: 0.2647103	best: 0.2647103 (1200)	total: 5m 57s	remaining: 8m 55s
1400:	learn: 0.2500381	test: 0.2642608	best: 0.2642590 (1399)	total: 6m 57s	remaining: 7m 56s
1600:	learn: 0.2478357	test: 0.2639169	best: 0.2639169 (1600)	total: 7m 56s	remaining: 6m 56s
1800:	learn: 0.2457638	test: 0.2636606	best: 0.2636606 (1800)	total: 8m 54s	remaining: 5m 55s
2000:	learn: 0.2438275	test: 0.2634272	best: 0.2634271 (1997)	total: 9m

0:	learn: 1.0525346	test: 1.0930330	best: 1.0930330 (0)	total: 264ms	remaining: 13m 12s
200:	learn: 0.2762191	test: 0.2699301	best: 0.2699301 (200)	total: 59s	remaining: 13m 41s
400:	learn: 0.2679247	test: 0.2652662	best: 0.2652662 (400)	total: 1m 53s	remaining: 12m 18s
600:	learn: 0.2632503	test: 0.2636475	best: 0.2636428 (599)	total: 2m 52s	remaining: 11m 27s
800:	learn: 0.2597162	test: 0.2627159	best: 0.2627159 (800)	total: 3m 48s	remaining: 10m 26s
1000:	learn: 0.2567107	test: 0.2621286	best: 0.2621260 (991)	total: 4m 44s	remaining: 9m 29s
1200:	learn: 0.2540149	test: 0.2615458	best: 0.2615453 (1198)	total: 5m 42s	remaining: 8m 32s
1400:	learn: 0.2516891	test: 0.2612062	best: 0.2611991 (1398)	total: 6m 38s	remaining: 7m 35s
1600:	learn: 0.2494633	test: 0.2608941	best: 0.2608916 (1599)	total: 7m 40s	remaining: 6m 42s
1800:	learn: 0.2474827	test: 0.2606552	best: 0.2606552 (1800)	total: 8m 41s	remaining: 5m 47s
2000:	learn: 0.2456036	test: 0.2604448	best: 0.2604448 (2000)	total: 9m 42

In [8]:
# Step 8: SLSQP Convex Weight Optimization & Ensemble Evaluation
print("==================================================")
print(" STEP 8: OUT-OF-FOLD EVALUATION & SLSQP OPTIMAL BLENDING")
print("==================================================")

oof_list = [
    np.clip(oof_lgb, 0.0, None),
    np.clip(oof_xgb, 0.0, None),
    np.clip(oof_cb, 0.0, None),
    np.clip(oof_nn, 0.0, None)
]
test_preds_list = [
    np.clip(preds_lgb, 0.0, None),
    np.clip(preds_xgb, 0.0, None),
    np.clip(preds_cb, 0.0, None),
    np.clip(preds_nn, 0.0, None)
]
model_names = ['LightGBM (Log1p)', 'XGBoost (Log1p)', 'CatBoost (Log1p)', 'PyTorch Temporal Attention']

print("--- Individual Model Out-Of-Fold (OOF) Raw RMSE Scores ---")
for name, oof in zip(model_names, oof_list):
    score = np.sqrt(mean_squared_error(y_raw, oof))
    print(f"  -> {name:30s} | Out-Of-Fold RMSE: {score:.5f} ug/m3")

def optimize_blend_weights(oof_preds_list, y_true):
    num_models = len(oof_preds_list)
    def loss_func(weights):
        w = np.array(weights)
        w = w / np.sum(w)
        blend = sum(w[i] * oof_preds_list[i] for i in range(num_models))
        return np.sqrt(mean_squared_error(y_true, blend))
    
    init_weights = np.ones(num_models) / num_models
    bounds = [(0.0, 1.0)] * num_models
    constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
    res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x / np.sum(res.x)

print("\n--- Solving SLSQP Convex Optimization for Ensemble Weights ---")
opt_w = optimize_blend_weights(oof_list, y_raw)
for name, w in zip(model_names, opt_w):
    print(f"  -> Optimal Weight [{name:30s}] = {w:6.4f} ({w*100:5.2f}%)")

final_oof = sum(opt_w[i] * oof_list[i] for i in range(len(oof_list)))
final_rmse = np.sqrt(mean_squared_error(y_raw, final_oof))

print("\n" + "*"*65)
print(f"*** SOTA HYBRID ENSEMBLE OUT-OF-FOLD RMSE: {final_rmse:.5f} ug/m3 ***")
print("*"*65)

final_test_preds = sum(opt_w[i] * test_preds_list[i] for i in range(len(test_preds_list)))
final_test_preds = np.clip(final_test_preds, a_min=0.0, a_max=None)

 STEP 8: OUT-OF-FOLD EVALUATION & SLSQP OPTIMAL BLENDING
--- Individual Model Out-Of-Fold (OOF) Raw RMSE Scores ---
  -> LightGBM (Log1p)               | Out-Of-Fold RMSE: 17.17045 ug/m3
  -> XGBoost (Log1p)                | Out-Of-Fold RMSE: 17.46546 ug/m3
  -> CatBoost (Log1p)               | Out-Of-Fold RMSE: 17.48606 ug/m3
  -> PyTorch Temporal Attention     | Out-Of-Fold RMSE: 19.20087 ug/m3

--- Solving SLSQP Convex Optimization for Ensemble Weights ---
  -> Optimal Weight [LightGBM (Log1p)              ] = 0.6244 (62.44%)
  -> Optimal Weight [XGBoost (Log1p)               ] = 0.0371 ( 3.71%)
  -> Optimal Weight [CatBoost (Log1p)              ] = 0.2514 (25.14%)
  -> Optimal Weight [PyTorch Temporal Attention    ] = 0.0871 ( 8.71%)

*****************************************************************
*** SOTA HYBRID ENSEMBLE OUT-OF-FOLD RMSE: 17.08107 ug/m3 ***
*****************************************************************


In [9]:
# Step 9: Save GPU Ada Submission CSV
print("==================================================")
print(" STEP 9: SUBMISSION CSV GENERATION (GPU ADA MODE)")
print("==================================================")

sub_df = pd.DataFrame({
    'id': df_test['id'],
    'PM2.5': final_test_preds
})

sub_filename = 'submission_gpu_ada.csv'
sub_df.to_csv(sub_filename, index=False)
print(f"[SUCCESS] Saved {len(sub_df):,} predictions to '{sub_filename}'")
print("\nFinal Submission Summary & Stats:")
print(f"   Total Test IDs:       {len(sub_df):,}")
print(f"   Min Predicted PM2.5:  {sub_df['PM2.5'].min():.2f} ug/m3")
print(f"   Max Predicted PM2.5:  {sub_df['PM2.5'].max():.2f} ug/m3")
print(f"   Mean Predicted PM2.5: {sub_df['PM2.5'].mean():.2f} ug/m3")
print(f"   Std Predicted PM2.5:  {sub_df['PM2.5'].std():.2f} ug/m3")
print(f"   Null Count:           {sub_df['PM2.5'].isnull().sum()}")

display(sub_df.head(10))

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(sub_filename)

 STEP 9: SUBMISSION CSV GENERATION (GPU ADA MODE)
[SUCCESS] Saved 4,103 predictions to 'submission_gpu_ada.csv'

Final Submission Summary & Stats:
   Total Test IDs:       4,103
   Min Predicted PM2.5:  3.26 ug/m3
   Max Predicted PM2.5:  508.67 ug/m3
   Mean Predicted PM2.5: 75.66 ug/m3
   Std Predicted PM2.5:  78.44 ug/m3
   Null Count:           0


,id,PM2.5
0,test_00001,187.123332
1,test_00002,157.073614
2,test_00003,407.533048
3,test_00004,61.058779
4,test_00005,48.088141
5,test_00006,117.400023
6,test_00007,10.292947
7,test_00008,7.954767
8,test_00009,50.911281
9,test_00010,129.301408
